# Demo 02 : A first look at LangGraph

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import csv
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

db = sqlite3.connect(data("demo-world/service.db"), check_same_thread=False)
db.row_factory = sqlite3.Row

SECTIONS = []
for path in sorted(Path(data_folder("demo-world/manuals")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]
TICKETS = [", ".join(f"{k}={v}" for k, v in t.items())
           for t in csv.DictReader(open(data("demo-world/tickets.csv"), encoding="utf-8"))]


def rows(sql, *args):
    return [dict(r) for r in db.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]


print("sections:", len(SECTIONS), " tickets:", len(TICKETS),
      " buses:", len(rows("SELECT * FROM buses")), " parts:", len(rows("SELECT * FROM parts")))

sections: 82  tickets: 43  buses: 10  parts: 13


## a · first-graph

A graph of three plain functions, with no model anywhere. A driver's report
names a bus and an error code ; the graph looks up the bus, reads the first
action from the manual, and sends the bus to the workshop or fixes it on
site.

In [3]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class Report(TypedDict, total=False):
    fleet_no: str
    error_code: str
    model: str
    action: str
    where: str


def look_up_bus(state: Report) -> dict:
    return {"model": rows("SELECT model FROM buses WHERE fleet_no = ?", state["fleet_no"])[0]["model"]}


def read_manual(state: Report) -> dict:
    manual = next(s for s in SECTIONS if s.startswith(f"# {state['model']} error codes"))
    row = next(line for line in manual.splitlines() if line.startswith(f"| {state['error_code']} |"))
    return {"action": row.split("|")[3].strip()}


def route(state: Report) -> Literal["workshop", "on_site"]:
    return "workshop" if "workshop" in state["action"] else "on_site"


builder = StateGraph(Report)
builder.add_node("look_up_bus", look_up_bus)
builder.add_node("read_manual", read_manual)
builder.add_node("workshop", lambda state: {"where": "workshop"})
builder.add_node("on_site", lambda state: {"where": "on site"})
builder.add_edge(START, "look_up_bus")
builder.add_edge("look_up_bus", "read_manual")
builder.add_conditional_edges("read_manual", route)
builder.add_edge("workshop", END)
builder.add_edge("on_site", END)
graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	look_up_bus(look_up_bus)
	read_manual(read_manual)
	workshop(workshop)
	on_site(on_site)
	__end__([<p>__end__</p>]):::last
	__start__ --> look_up_bus;
	look_up_bus --> read_manual;
	read_manual -.-> on_site;
	read_manual -.-> workshop;
	on_site --> __end__;
	workshop --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



The state is a dictionary that every node reads. Each node returns only the
keys it changes, and LangGraph merges them. The conditional edge reads the
state to pick the next node.

In [4]:
for report in [{"fleet_no": "CD12-2024-0098", "error_code": "F-CD-202"},
               {"fleet_no": "CE12-2021-0117", "error_code": "F-CE-305"}]:
    result = graph.invoke(report)
    print(f"{result['fleet_no']} {result['error_code']} -> {result['where']:8} {result['action']}")

CD12-2024-0098 F-CD-202 -> workshop withdraw the bus from service ; call the diesel specialist ; fuel pump replacement needs a workshop bay
CE12-2021-0117 F-CE-305 -> on site  inspect and replace the door actuator


`stream` shows the same run one node at a time, with the update each node
returned.

In [5]:
for step in graph.stream({"fleet_no": "MM09-2025-2350", "error_code": "F-MM-210"}):
    print(step)

{'look_up_bus': {'model': 'MM-09'}}
{'read_manual': {'action': 'replace the auxiliary battery'}}
{'on_site': {'where': 'on site'}}


## b · first-multi-agent

We put two agents behind a supervisor, each with its own tools and prompt.
The supervisor calls them as tools and reads their returned answers.

In [6]:
@tool
def bus_info(fleet_no: str) -> str:
    """Model, depot and firmware of one bus, by fleet number such as CE12-2021-0117."""
    return str(rows("SELECT * FROM buses WHERE fleet_no = ?", fleet_no.strip().upper()) or "no such bus")


@tool
def part_stock(part_no: str) -> str:
    """Name, units in stock, price and depot of one spare part, by part number such as CE-CF-02."""
    return str(rows("SELECT * FROM parts WHERE part_no = ?", part_no.strip().upper()) or "no such part")


@tool
def parts_for_model(model: str) -> str:
    """Every spare part for a bus model such as CE-12, with units in stock."""
    return str(rows("SELECT * FROM parts WHERE model = ?", model.strip().upper()))


@tool
def search_manuals(query: str) -> str:
    """Search the service manuals and bulletins. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

from langchain_core.callbacks import BaseCallbackHandler


class Meter(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input = self.output = 0

    def on_llm_end(self, response, **kwargs):
        usage = response.generations[0][0].message.usage_metadata or {}
        self.calls += 1
        self.input += usage.get("input_tokens", 0)
        self.output += usage.get("output_tokens", 0)

    def __str__(self):
        return f"{self.calls} calls, {self.input:,} input tokens, {self.output:,} output tokens"


def correct(answer, expected):
    text = answer.lower()
    return all(any(alt in text for alt in item.split("|")) for item in expected)

In [7]:
from langchain.agents import create_agent

parts_agent = create_agent(llm, tools=[bus_info, part_stock, parts_for_model],
                           system_prompt="You answer questions about spare parts and stock. "
                                         "Use bus_info to resolve a fleet number to a model. "
                                         "Use parts_for_model to find a part number from a component name. "
                                         "Fault codes are not part numbers. Answer from the tools. Be brief.")
manuals_agent = create_agent(llm, tools=[search_manuals],
                             system_prompt="You answer questions from the service manuals. Be brief.")


@tool
def ask_parts_agent(question: str) -> str:
    """Ask the parts agent about part numbers and stock."""
    reply = parts_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return reply["messages"][-1].content


@tool
def ask_manuals_agent(question: str) -> str:
    """Ask the manuals agent about error codes, procedures and maintenance."""
    reply = manuals_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return reply["messages"][-1].content


supervisor = create_agent(llm, tools=[ask_parts_agent, ask_manuals_agent],
                          system_prompt="You run the Veenhaven Bus service desk. Ask the manuals agent "
                                        "first ; pass the component it names to the parts agent. "
                                        "Then answer the mechanic briefly.")

One question that needs both agents. The model decides how to split it, so
the order and wording of the delegated questions vary between runs.

In [8]:
question = "Bus CE12-2021-0117 shows F-CE-110. What part do I need and is it in stock?"
meter = Meter()
result = supervisor.invoke({"messages": [{"role": "user", "content": question}]}, {"callbacks": [meter]})

for m in result["messages"]:
    for call in getattr(m, "tool_calls", []):
        print(f"{call['name']:18} Q: {call['args']['question']}")
    if m.type == "tool":
        print(f"{m.name:18} A: {m.content[:200]}\n")
print("final:", result["messages"][-1].content)
print("correct:", correct(result["messages"][-1].content, ["ce-cf-02|cooling fan", "6|six"]))
print(meter)

ask_manuals_agent  Q: For bus CE12-2021-0117, what does fault code F-CE-110 indicate, and which component/part should be checked or replaced?
ask_manuals_agent  A: F-CE-110 indicates a **traction system over-temperature**. Check the **traction cooling fan** and replace it if its airflow is below 80%.

ask_parts_agent    Q: Is the traction cooling fan for bus CE12-2021-0117 in stock? Please provide the correct part number if available.
ask_parts_agent    A: Yes. The traction cooling fan is in stock: **part number CE-CF-02**, with **6 units** available.

final: F-CE-110 indicates traction-system overheating. Check the traction cooling fan; replace it if airflow is below 80%. Fan part **CE-CF-02** is in stock (6 available).
correct: True
8 calls, 2,339 input tokens, 294 output tokens


We inspect the delegated questions to see what each agent received. Each
specialist starts a fresh conversation ; the supervisor can include
earlier findings in a later task. Asked both at once, the parts agent gets
no component name and cannot find the part, so the prompt orders the
manuals agent first. The meter includes nested model calls.

## Things to try

Optional.

- Add a branch to the first graph for codes that are not in the manual
- Ask the supervisor a question only one agent can answer, and count the calls
- Print `parts_agent.get_graph().draw_mermaid()` : an agent is a graph too